In [16]:
import torch
import torch.nn as nn
from torchvision import models

model = models.mobilenet_v3_small(pretrained=False)
model.classifier = nn.Sequential(
    nn.Linear(576, 1024),
    nn.Hardswish(),
    nn.Dropout(p=0.2),
    nn.Linear(1024, 256),
)
model.load_state_dict(torch.load('best_model.pth', map_location=torch.device('cpu')))

<All keys matched successfully>

In [18]:
import onnx

m = onnx.load("ver2_model.onnx")
print("Opset imports:", [(o.domain, o.version) for o in m.opset_import])

Opset imports: [('', 18)]


In [17]:
model.eval()
dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    model,
    dummy,
    'ver2_model.onnx',
    input_names=['input'],
    output_names=['output'],
    #dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
    do_constant_folding=True,
    opset_version=13
)

print("Model has been converted to ONNX format and saved as 'model.onnx'.")

W0303 15:23:45.853000 6752 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0303 15:23:46.213000 6752 Lib\site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0303 15:23:46.213000 6752 Lib\site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', s

[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 13).


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 13 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "d:\THAI\PROJECT\CBIR_ON_EDGE_DEVICE\.venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\THAI\PROJECT\CBIR_ON_EDGE_DEVICE\.venv\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "d:\THAI\PROJECT\CBIR_ON_EDGE_DEVICE\.venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\THAI\PROJECT\CBIR_ON_EDGE_DEVICE\.venv\Lib\site-packages\onnx\version_converter.py", line 38, in convert_version
    converted_model_str = C.convert_version(model_str, target_versio

Applied 68 of general pattern rewrite rules.
Model has been converted to ONNX format and saved as 'model.onnx'.


In [19]:
!onnx2tf -i ver2_model.onnx -o ver2_tf_cbir_256


Model optimizing started ============================================================
Traceback (most recent call last):
  File "d:\THAI\PROJECT\CBIR_ON_EDGE_DEVICE\.venv\Lib\site-packages\onnx2tf\onnx2tf.py", line 716, in convert
    result = subprocess.check_output(
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Windows\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 466, in check_output
    return run(*popenargs, stdout=PIPE, timeout=timeout, check=True,
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Windows\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Windows\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1024, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Windows\AppData\Local\Programs\Python\Python311\Lib\subprocess.py


I0000 00:00:1772526295.165891   11604 devices.cc:76] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0 (Note: TensorFlow was not compiled with CUDA or ROCm support)
I0000 00:00:1772526295.166079   11604 single_machine.cc:374] Starting new session
W0000 00:00:1772526295.820256   11604 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1772526295.820304   11604 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1772526296.357931   11604 devices.cc:76] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0 (Note: TensorFlow was not compiled with CUDA or ROCm support)
I0000 00:00:1772526296.358111   11604 single_machine.cc:374] Starting new session
W0000 00:00:1772526297.134560   11604 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1772526297.134604   11604 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.


In [20]:
import tensorflow as tf
import tf_keras
import onnx
import onnxruntime
import onnx_graphsurgeon
import onnx2tf
import sng4onnx
import numpy as np

print("=== TensorFlow stack ===")
print("tensorflow:", tf.__version__)
print("tf_keras:", tf_keras.__version__)

print("\n=== ONNX stack ===")
print("onnx:", onnx.__version__)
print("onnxruntime:", onnxruntime.__version__)
print("onnx_graphsurgeon:", onnx_graphsurgeon.__version__)
print("onnx2tf:", onnx2tf.__version__)
print("sng4onnx:", sng4onnx.__version__)

print("\n=== Core ===")
print("numpy:", np.__version__)

=== TensorFlow stack ===
tensorflow: 2.19.0
tf_keras: 2.19.0

=== ONNX stack ===
onnx: 1.17.0
onnxruntime: 1.24.2
onnx_graphsurgeon: 0.5.8
onnx2tf: 1.28.8
sng4onnx: 2.0.1

=== Core ===
numpy: 1.26.4
